## Snippet 1- Download helper for chapter support code
Before the chapter can load and analyze NSFG survey data, the notebook first defines a tiny helper that downloads a local copy of the shared `thinkstats.py` support module if it is missing. This matches the chapter’s opening workflow: get the environment ready, then move on to reading, checking, and exploring the data. 

In [ ]:
""" 
Reading the Data
Context: this helper makes sure the notebook has the local support file it needs before later cells import from it
Plain-English: it checks whether a file already exists, and if not, downloads it from the book's GitHub repository

Use of in-built libraries os and urllib
"""

from os.path import basename, exists  # import one function to pull a filename from an online URL, and one to check whether that file is already present
# os stand operating system module
# path.py contains basename and exists
# exists function does ... 
# basename function creates the local filename by taking the last path segment from the URL with 1 parameter URL (string datatype)


def download(url):  # define a reusable helper function that takes a single parameter, URL str (a string representing the URL of the file to download)
    filename = basename(url)  # create the local filename by taking the last path segment from the URL
    print("Downloaded file from URL")
    if not exists(filename):  # only download when that filename is not already in the current working folder
        from urllib.request import urlretrieve  # import the downloader here so it is only needed if a download actually happens

        local, _ = urlretrieve(url, filename)  # fetch the file from the URL and save it locally under the chosen filename
        print("Downloaded " + local)  # report which local file was written so the user can confirm the side effect
# url retrieve function does fetch the file from the URL and save it locally under the chosen filename with 2 parameters (string)
# return object of urlretrieve is 2 items which are (filename, headers) where headers is metadata from the server

download("https://github.com/AllenDowney/ThinkStats/raw/v3/nb/thinkstats.py")  # fetch the shared helper module used by later notebook cells

"""
Interpretation: running this cell either creates a new local file named `thinkstats.py` or leaves the existing copy in place. 
If the cell prints a download message, a file was just written to disk; 
if it prints nothing, the file was already available and memory now contains the `download` function for later reuse. 
A common error here is a network or permission failure, which usually means the notebook cannot reach the URL or cannot write files in the current environment.


* Note you can import py files from your Github or somebody else's, 
by navigating to the file online and clicking raw button and then copying URL from it so you can execute download function to put new file in your cwd

"""

print()


Downloaded file from URL
Downloaded thinkstats.py



In [16]:
"""
We should always verify file correctness (file vs folder), not just existence, before using it in data. 
If we accidentally created a folder workflows like "2002FemPreg.dct/" or filename already exists as a directory "urlretrieve(url, filename)" , Python does NOT overwrite it.
And exists() returns True for BOTH files & folders. But it should have checked specifically for a file. 
"""

from os.path import basename, isfile, isdir  
# basename extracts filename; 
# isfile checks real files; 
# isdir detects incorrect folder state

import shutil  # needed to remove directories safely

def download(url):  # define helper that safely downloads a file from a URL
    filename = basename(url)  # extract local filename from URL
    print("Downloaded file from URL")
    
    if isdir(filename):  # check if a folder exists where a file should be
        shutil.rmtree(filename)  # remove the folder to fix incorrect state
    
    if not isfile(filename):  # only download if a valid file does not already exist
        from urllib.request import urlretrieve  # import downloader only when needed
        
        local, _ = urlretrieve(url, filename)  # download file and save locally
        print("Downloaded " + local)  # confirm successful download

download("https://github.com/AllenDowney/ThinkStats/raw/v3/nb/thinkstats.py")  # ensure support module is available

Downloaded file from URL


## Snippet 2- Ensure empirical distribution library is available
Before working with distributions later in the chapter, the notebook ensures that the `empiricaldist` library is available. This follows the book’s practical approach: install lightweight tools on demand so distribution analysis can run smoothly without manual setup.

In [ ]:
"""
# Data Loading
# Context: this snippet ensures a required library is available before later cells rely on it for distribution and stata-style dictionary related computations
# Plain-English: it tries to import a library and installs it automatically if it is missing
# Reappears: yes; on-demand installation patterns appear in other chapters and projects
"""

try:
    import empiricaldist  # attempt to load the empiricaldist package into memory
except ImportError:  # catch the specific error raised when the package is not installed
    %pip install empiricaldist  # use notebook magic to install the package into the current environment
try:
    import statadict  # attempt to load statadict for reading Stata-style data dictionary files
except ImportError:  # run this block if the package is not installed
    %pip install statadict  # install statadict into the current environment

"""
# Interpretation: This cell either loads the both empiricaldist and statadict package or installs it if it is not already available.
# If the packages are already installed, the cell runs silently with no output.
# If installation is needed, pip output will appear showing download and install progress.
# After execution, both empiricaldist and statadict becomes available for import in later cells.
# A common issue is network failure during installation, which prevents pip from downloading the package.
# Another issue is environment restrictions where package installation is not permitted.
"""
print()

## Snippet 3- Import core libraries and helper utilities
With the helper module in place, the notebook now loads the main libraries and helper tools used throughout the chapter. This prepares the environment for working with data tables, numerical arrays, visualizations, and formatted outputs as the exploratory analysis begins.

In [7]:
# Data Loading
# Context: this snippet prepares all core libraries and helper utilities that later cells depend on for analysis and visualization
# Plain-English: it loads libraries for data handling, numerical computation, plotting, and display formatting
# Real-world: this combined import pattern is common in analysis notebooks to centralize all dependencies at the start
# Reappears: yes; these libraries and helpers are used repeatedly across this chapter and later chapters

import numpy as np  # load NumPy as np for numerical operations on arrays
import pandas as pd  # load pandas as pd for working with tabular data using DataFrames
import matplotlib.pyplot as plt  # load plotting interface as plt for creating charts

from IPython.display import HTML  # import HTML utility to render formatted output in notebook cells
from thinkstats import decorate  # import decorate helper to standardize plot labels and styling

# Interpretation
# This cell loads five tools into memory: np, pd, plt, HTML, and decorate.
# There is no visible output when it runs successfully, which is expected.
# These imports enable data manipulation (pandas), numerical work (NumPy), plotting (Matplotlib),
# rich display (HTML), and consistent plot formatting (decorate).
# A common error is ModuleNotFoundError for thinkstats, indicating the support file is missing.
# Another possible issue is missing libraries, which would raise ImportError and require installation.

## Snippet 4- Download NSFG dataset files for analysis
Before loading the dataset, the notebook ensures that the raw NSFG data files are available locally. These files include the data dictionary and compressed survey data, which the chapter uses to construct the DataFrame for analysis.

In [8]:
# Data Loading
# Context: this snippet ensures the required dataset files exist locally before the data-loading function depends on them
# Plain-English: it downloads the survey data files if they are not already present
# Real-world: this pattern appears in data pipelines where raw datasets are fetched before processing begins
# Reappears: yes; dataset download and setup steps recur in other data-driven chapters

download("https://github.com/AllenDowney/ThinkStats/raw/v3/data/2002FemPreg.dct")  # download the data dictionary file describing column structure
download("https://github.com/AllenDowney/ThinkStats/raw/v3/data/2002FemPreg.dat.gz")  # download the compressed raw dataset file

# Interpretation
# This cell checks whether each dataset file exists locally and downloads it only if missing.
# If downloads occur, messages will print confirming the saved files.
# If no output appears, the files were already present, which is expected in some environments.
# After this step, the required data files are available for ReadFemPreg to load into a DataFrame.
# A common error is a network issue preventing download, or file permission issues blocking writes to the directory.

Downloaded 2002FemPreg.dct
Downloaded 2002FemPreg.dat.gz


## Snippet 5- Store the dataset filenames in variables
With the raw NSFG files available locally, the notebook assigns their names to variables. This makes the next data-reading step clearer because later code can refer to the dictionary file and data file by purpose rather than repeating raw strings.

In [9]:
# Data Loading
# Context: this snippet creates named references to the two downloaded files so the reader function can use them cleanly in the next step
# Plain-English: it stores the dictionary filename and compressed data filename in variables
# Reappears: using variables for file paths comes back throughout data-loading workflows

dct_file = "2002FemPreg.dct"  # create a variable holding the local filename of the Stata-style data dictionary
dat_file = "2002FemPreg.dat.gz"  # create a variable holding the local filename of the compressed raw data file

# Interpretation: this cell creates two string variables in memory, `dct_file` and `dat_file`. There is no visible output, which is normal. Later cells will pass these names into the data-reading function instead of repeating the filenames directly. A common error here is a typo in a filename string, which usually leads to a file-not-found error only when a later cell tries to open the file.
import os

print("Exists:", os.path.exists("2002FemPreg.dct"))
print("Is file:", os.path.isfile("2002FemPreg.dct"))
print("Is dir:", os.path.isdir("2002FemPreg.dct"))

Exists: True
Is file: True
Is dir: False


## Snippet 6- Define function to read NSFG data using the data dictionary
Instead of loading the dataset directly, the notebook defines a reusable function that reads the data dictionary and applies it to parse the raw fixed-width file correctly. This reflects the book’s approach of understanding structure first, then building a clean and repeatable data-loading step.

In [ ]:
# Data Loading
# Context: this snippet defines how to read the dataset and then uses that logic to load it into memory
# Plain-English: it builds a function to parse structured raw data and immediately applies it to create a DataFrame
# Reappears: yes; custom loaders and immediate usage appear throughout data workflows

from statadict import parse_stata_dict  # import parser for reading Stata-style dictionary files


def read_stata(dct_file, dat_file):  # define function to read dataset using dictionary metadata
    stata_dict = parse_stata_dict(dct_file)  # extract column names and positions from dictionary
    
    resp = pd.read_fwf(  # read fixed-width formatted file into a DataFrame
        dat_file,  # path to compressed dataset file
        names=stata_dict.names,  # assign column names from dictionary
        colspecs=stata_dict.colspecs,  # assign column boundaries from dictionary
        compression="gzip",  # specify gzip compression for the file
    )
    
    return resp  # return the resulting DataFrame

preg = read_stata(dct_file, dat_file)  # call the function to load dataset into variable preg
print(preg)


# Interpretation
# This cell first defines the read_stata function, then immediately uses it to create a DataFrame named preg.
# The function ensures the raw data is correctly interpreted using metadata from the dictionary file.
# The resulting DataFrame contains structured survey data ready for analysis.
# No output is shown unless explicitly displayed.
# Common errors include missing files, incorrect paths, or issues with dictionary parsing leading to misaligned data.


       caseid  pregordr  howpreg_n  howpreg_p  moscurrp  nowprgdk  pregend1  \
0           1         1        NaN        NaN       NaN       NaN       6.0   
1           1         2        NaN        NaN       NaN       NaN       6.0   
2           2         1        NaN        NaN       NaN       NaN       5.0   
3           2         2        NaN        NaN       NaN       NaN       6.0   
4           2         3        NaN        NaN       NaN       NaN       6.0   
...       ...       ...        ...        ...       ...       ...       ...   
13588   12571         1        NaN        NaN       NaN       NaN       6.0   
13589   12571         2        NaN        NaN       NaN       NaN       3.0   
13590   12571         3        NaN        NaN       NaN       NaN       3.0   
13591   12571         4        NaN        NaN       NaN       NaN       6.0   
13592   12571         5        NaN        NaN       NaN       NaN       6.0   

       pregend2  nbrnaliv  multbrth  ...  poverty_i

## Snippet 7- Inspect dataset shape, columns, and sample rows
After loading the dataset, the notebook performs a quick structural check by looking at its size, column names, a small sample of rows and then focuses on a specific variable, `pregordr`, which represents the pregnancy order. This aligns with the chapter’s goal of identifying and understanding relevant variables before analysis.

In [ ]:
# Data Loading
# Context: this snippet validates the dataset structure so later analysis is based on correctly loaded data
# Plain-English: it checks how many rows and columns exist, what the column names are, and previews sample data
# Reappears: yes; structural inspection is a standard first step in exploratory data analysis

preg.shape  # read attribute that returns a tuple (number of rows, number of columns)

print(preg.shape)

preg.columns  # read attribute that lists all column names in the DataFrame

print(preg.columns)

preg.head()  # call method to display the first 5 rows for a quick data preview

pregordr = preg["pregordr"]  # select the pregordr column (pregnancy order) from the DataFrame and store it separately

pregordr.head()  # display first 5 values of the pregordr column

# Interpretation
# preg.shape outputs a tuple showing dataset size: (rows, columns).
# preg.columns outputs an index of column names, which helps verify variables are correctly parsed.
# preg.head() displays the first few rows so you can visually inspect values and structure.
# If shape shows unexpected dimensions, it may indicate parsing issues.
# If columns look incorrect or misaligned, the dictionary parsing step may have failed.
# If head() shows garbled or shifted data, column specifications may not match the raw file format.
# pregordr extracts one column as a pandas Series representing pregnancy order.
# pregordr.head() shows the first few values of this variable for inspection.
# If pregordr does not exist in columns, a KeyError will occur, indicating parsing or naming issues

(13593, 243)
Index(['caseid', 'pregordr', 'howpreg_n', 'howpreg_p', 'moscurrp', 'nowprgdk',
       'pregend1', 'pregend2', 'nbrnaliv', 'multbrth',
       ...
       'poverty_i', 'laborfor_i', 'religion_i', 'metro_i', 'basewgt',
       'adj_mod_basewgt', 'finalwgt', 'secu_p', 'sest', 'cmintvw'],
      dtype='str', length=243)


0    1
1    2
2    1
3    2
4    3
Name: pregordr, dtype: int64